# 01 — Matched target-load latency

This view compares run-level latency percentiles at the matched target load. It does not reconstruct a CDF from summary points or establish saturation. N is runs, units are milliseconds, and diagnostic/thesis and uncertainty labels are explicit. Missing systems remain PENDING, never zero.


In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, passed_artifacts, pending_record, percentile_rows
from wafer_analysis.paths import resolve_result_batch

def resolve(experiment, env_name):
    try:
        return resolve_result_batch(experiment, diagnostic_path=os.environ.get(env_name))
    except (FileNotFoundError, RuntimeError, ValueError):
        return None

batch=resolve('e-perf-2','E_PERF_2_DIR')
df=pd.DataFrame() if batch is None else percentile_rows(batch)
expected={'wafer','native','ekuiper'}
rows=[]
for system in sorted(expected):
    values=df[df.condition==system] if not df.empty else pd.DataFrame()
    if values.empty: rows.append(pending_record(system,'no passed percentile leaf','milliseconds'))
    else: rows.append({'question':system,'status':'READY','value':values.p99_ns.median()/1e6,'units':'p99 milliseconds','reason':evidence_label(len(values),'milliseconds',False),'thesis_evidence':False})
out=pd.DataFrame(rows)
print(evidence_label(len(out[out.status=='READY']), 'milliseconds', False))
display(out)
ready=out[out.status=='READY']
if not ready.empty:
    ax=ready.plot.bar(x='question',y='value',legend=False); ax.set_ylabel('Median run p99 (ms)'); ax.set_title('Matched target-load latency — diagnostic')
